In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

In [ ]:
cd drive/MyDrive/Datalab/practice

/content/drive/MyDrive/Datalab/practice


In [ ]:
import os
import shutil
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
df = pd.read_csv("../data/daily-median-income.csv")
df = df.drop(columns=["Code"])

df = df.rename(columns={
    "Median (2021 prices)": "value"
})

global_min_year = df["Year"].min()
global_max_year = df["Year"].max()

countries = df["Entity"].unique()
years = range(global_min_year, global_max_year + 1)

full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["Entity", "Year"]
)

df_full = (
    df
    .set_index(["Entity", "Year"])
    .reindex(full_index)
    .reset_index()
)


df_full = df_full.sort_values(['Entity', 'Year'])


In [ ]:
df_full

,Entity,Year,value
0,Albania,1963,NaN
1,Albania,1964,NaN
2,Albania,1965,NaN
3,Albania,1966,NaN
4,Albania,1967,NaN
...,...,...,...
12154,Zimbabwe,2021,NaN
12155,Zimbabwe,2022,NaN
12156,Zimbabwe,2023,NaN
12157,Zimbabwe,2024,NaN


In [ ]:
# 1. Pivot 처리: (193개국, 31년) 형태의 2D Matrix 생성
pivot_df = df.pivot(index='Entity', columns='Year', values='value')

# 2. 정규화 (금융 데이터는 스케일 차이가 커서 필수입니다)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# 시간축 흐름을 유지하며 정규화하기 위해 T(전치)를 활용
scaled_vals = scaler.fit_transform(pivot_df.values.T).T

In [ ]:
# 3. 3차원 변환: (국가수, 연도수, 1)
# 여기서 마지막 '1'은 측정하는 지표가 'value' 하나뿐이라는 뜻입니다.
X = scaled_vals.reshape(pivot_df.shape[0], pivot_df.shape[1], 1)

print(f"최종 입력 데이터 X의 모양: {X.shape}")
# 결과가 (193, 31, 1) 또는 (국가수, 연도수, 1)이 나와야 합니다.

최종 입력 데이터 X의 모양: (193, 63, 1)


In [ ]:
!pip install pypots

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.8/749.8 kB 15.2 MB/s eta 0:00:00


In [ ]:
from pypots.imputation import TimesNet

# 1. 모델 재정의 (에러 방지용 설정)
timesnet = TimesNet(
    n_steps=63,           # 연도 수
    n_features=1,         # 변수 개수
    n_layers=2,
    top_k=3,              # 중요! 3에서 1로 줄여서 복잡한 주기성 계산을 단순화합니다.
    d_model=32,           # 소수 데이터이므로 차원을 조금 줄여 안정성을 높입니다.
    d_ffn=64,
    n_kernels=3,
    dropout=0.2,
    epochs=100,
    patience=10,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# 2. 다시 학습 시도
# X는 (193, 31, 1) 형태여야 합니다.
timesnet.fit({"X": X})

2025-12-28 16:57:29 [INFO]: Using the given device: cpu
2025-12-28 16:57:29 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
2025-12-28 16:57:29 [INFO]: Using customized MAE as the training loss function.
2025-12-28 16:57:29 [INFO]: Using customized MSE as the validation metric function.
2025-12-28 16:57:29 [INFO]: TimesNet initialized with the given hyperparameters, the number of trainable parameters: 287,553
2025-12-28 16:57:31 [INFO]: Epoch 001 - training loss (MAE): 0.5018
2025-12-28 16:57:33 [INFO]: Epoch 002 - training loss (MAE): 0.3478
2025-12-28 16:57:34 [INFO]: Epoch 003 - training loss (MAE): 0.2651
2025-12-28 16:57:36 [INFO]: Epoch 004 - training loss (MAE): 0.2136
2025-12-28 16:57:38 [INFO]: Epoch 005 - training loss (MAE): 0.1866
2025-12-28 16:57:41 [INFO]: Epoch 006 - training loss (MAE): 0.1597
2025-12-28 16:57:43 [INFO]: Epoch 007 - training loss (MAE): 0.1397
2025-12-28 16:57:45 [INFO]: Epoch 008 - training loss (MAE): 0.1396
20

In [ ]:
from pypots.imputation import Informer
import torch

# 1. 모델 정의
informer = Informer(
    n_steps=31,           # 연도 수
    n_features=1,         # 변수 개수 (국가별 1개)
    n_layers=2,           # 인코더 레이어 수
    d_model=64,           # 임베딩 차원
    d_ffn=128,            # 피드포워드 차원
    n_heads=4,            # 멀티헤드 어텐션 개수
    dropout=0.2,
    epochs=100,
    patience=10,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# 2. 모델 학습
# X는 (193, 31, 1) 넘파이 배열
informer.fit({"X": X})

# 3. 보간 실행
predictions_inf = informer.predict({"X": X})

# 4. 결과 데이터 추출
imputed_data_inf = predictions_inf.get('imputation', predictions_inf.get('imputed_X'))

print("Informer 보간 완료!")